# 02 — FinTech Data Cleaning

**Author:** Ricardo Sanchez  

**Project:** Monetary Policy Shocks and Startup Funding Transitions (Seed → Series A)

**Date:** March 03, 2026 

**Data Source:** Crunchbase (2016 – 2023 Q3)  

**Sector:** FinTech

---

## Objective

Transform raw Crunchbase FinTech funding data into a clean, analysis-ready survival dataset that tracks each startup's time from Seed funding to Series A. The output is a single CSV with one row per firm, containing the event indicator, survival time, metro-region classification, and key covariates.

## Pipeline Overview

| Stage | Description |
|-------|-------------|
| **Part 1** | Merge raw seed funding packages (A + B) into a single file |
| **Part 2** | Build the survival dataset — collapse to one round per firm, define event/censoring, compute survival times |
| **Part 3** | Assign metro regions, attach covariates, label sector |
| **Part 4** | Select final columns and export |

## Inputs
- `data/raw/fintech/fintech_seed_funding_package_A.csv`
- `data/raw/fintech/fintech_seed_funding_package_B.csv`
- `data/raw/fintech/fintech_series_A_funding.csv`
- `data/cleaned/fintech_loc_map/fintech_city_to_metro.csv`

## Outputs
- `data/cleaned/cleaned_fintech/fintech_simple_survival.csv`

---

## Part 1 — Merge Raw Seed Funding Packages

Crunchbase exports were downloaded in two batches (package A and package B). This section sorts both by announcement date and concatenates them into a single seed funding file.

### 1.1 Setup & Load Raw Packages

In [2]:
import pandas as pd
from datetime import datetime

# Load raw Crunchbase seed funding exports (two download batches)
packA_df = pd.read_csv(r"..\data\raw\fintech\fintech_seed_funding_package_A.csv")
packB_df = pd.read_csv(r"..\data\raw\fintech\fintech_seed_funding_package_B.csv")

### 1.2 Inspect Date Ranges Before Merge

Verify that the two packages cover complementary or overlapping date ranges, and confirm chronological ordering after sort.

In [3]:
# Inspect date boundaries of each package before merging
print(f'Packet A first 3\n{packA_df['Announced Date'].head(3)}')
print(f'Packet A last 3\n{packA_df['Announced Date'].tail(3)} \n')
print(f'Packet B first 3\n{packB_df['Announced Date'].head(3)}')
print(f'Packet B last 3\n{packB_df['Announced Date'].tail(3)}')

Packet A first 3
0    2016-01-01
1    2016-01-01
2    2016-01-01
Name: Announced Date, dtype: object
Packet A last 3
997    2022-06-21
998    2022-06-23
999    2022-06-24
Name: Announced Date, dtype: object 

Packet B first 3
0    2024-12-17
1    2024-12-10
2    2024-12-06
Name: Announced Date, dtype: object
Packet B last 3
332    2022-06-28
333    2022-06-27
334    2022-06-26
Name: Announced Date, dtype: object


In [4]:
# Sorting both dataframes by "Announced Date" in ascending order
packA_df = packA_df.sort_values(by="Announced Date", ascending=True)
packB_df = packB_df.sort_values(by="Announced Date", ascending=True)

print(f'Packet A first 3\n{packA_df['Announced Date'].head(3)}')
print(f'Packet A last 3\n{packA_df['Announced Date'].tail(3)} \n')
print(f'Packet B first 3\n{packB_df['Announced Date'].head(3)}')
print(f'Packet B last 3\n{packB_df['Announced Date'].tail(3)}')

Packet A first 3
0    2016-01-01
1    2016-01-01
2    2016-01-01
Name: Announced Date, dtype: object
Packet A last 3
997    2022-06-21
998    2022-06-23
999    2022-06-24
Name: Announced Date, dtype: object 

Packet B first 3
334    2022-06-26
333    2022-06-27
332    2022-06-28
Name: Announced Date, dtype: object
Packet B last 3
2    2024-12-06
1    2024-12-10
0    2024-12-17
Name: Announced Date, dtype: object


In [5]:
# Concatenate teh two sorted packages into a single seed funding dataframe
fintech_merged_df = pd.concat([packA_df, packB_df], ignore_index=True)

print(f'Merged data first 3\n{fintech_merged_df['Announced Date'].head(3)} \n')
print(f'Merged data last 3\n{fintech_merged_df['Announced Date'].tail(3)}')

Merged data first 3
0    2016-01-01
1    2016-01-01
2    2016-01-01
Name: Announced Date, dtype: object 

Merged data last 3
1332    2024-12-06
1333    2024-12-10
1334    2024-12-17
Name: Announced Date, dtype: object


In [6]:
# Export merged seed funding data for downstream consumption
fintech_merged_df.to_csv(r"..\data\raw\fintech\fintech_seed_funding_merged.csv", index=False)

*Refer to `01_exploratory_data_analysis.ipynb` for exploratory analysis of the FinTech Seed and Series A funding datasets.*

---


## Part 2 — Build Survival Dataset

Construct a simple survival dataset suitable for Kaplan-Meier estimation and downstream hazard modeling. Each row represents one FinTech startup, tracking the time from its earliest Seed round to its earliest Series A round (or right-censoring if no Series A was observed).

### 2.0 Load Merged Data

In [7]:
# Load the merged seed and series A datasets
fintech_seed = pd.read_csv(r"..\data\raw\fintech\fintech_seed_funding_merged.csv")
fintech_seriesA = pd.read_csv(r"..\data\raw\fintech\fintech_series_A_funding.csv")

In [8]:
# Validate teh number of recoreds per colum - seed data
fintech_seed.notnull().count()

Transaction Name               1335
Transaction Name URL           1335
Organization Location          1335
Organization Name              1335
Organization Name URL          1335
Funding Type                   1335
Money Raised                   1335
Money Raised Currency          1335
Money Raised (in USD)          1335
Announced Date                 1335
Equity Only Funding            1335
Organization Revenue Range     1335
Organization Website           1335
Diversity Spotlight            1335
Number of Partner Investors    1335
Number of Investors            1335
dtype: int64

In [9]:
fintech_seriesA.count().notnull()

Transaction Name               True
Transaction Name URL           True
Organization Location          True
Organization Name              True
Organization Name URL          True
Funding Type                   True
Money Raised                   True
Money Raised Currency          True
Money Raised (in USD)          True
Announced Date                 True
Equity Only Funding            True
Organization Revenue Range     True
Organization Website           True
Diversity Spotlight            True
Number of Partner Investors    True
Number of Investors            True
dtype: bool

In [10]:
# Filter to correct funding round types only (guard against Crunchbase export contamination)
fintech_seed = fintech_seed[fintech_seed["Funding Type"] == "Seed"].copy()
fintech_seriesA = fintech_seriesA[fintech_seriesA["Funding Type"] == "Series A"].copy()


In [11]:
# Verify row counts after funding-type filter — seed
fintech_seed.count()

Transaction Name               1335
Transaction Name URL           1335
Organization Location          1335
Organization Name              1335
Organization Name URL          1335
Funding Type                   1335
Money Raised                   1335
Money Raised Currency          1335
Money Raised (in USD)          1335
Announced Date                 1335
Equity Only Funding            1335
Organization Revenue Range     1041
Organization Website           1332
Diversity Spotlight             345
Number of Partner Investors     572
Number of Investors            1092
dtype: int64

In [12]:
# Verify row counts after funding-type filter — Series A
fintech_seriesA.count()

Transaction Name               606
Transaction Name URL           606
Organization Location          606
Organization Name              606
Organization Name URL          606
Funding Type                   606
Money Raised                   606
Money Raised Currency          606
Money Raised (in USD)          606
Announced Date                 606
Equity Only Funding            606
Organization Revenue Range     557
Organization Website           605
Diversity Spotlight            155
Number of Partner Investors    463
Number of Investors            579
dtype: int64

In [ ]:
# Parse announcement dates to datetime for time-based operations
fintech_seed["Announced Date"] = pd.to_datetime(fintech_seed["Announced Date"])
fintech_seriesA["Announced Date"] = pd.to_datetime(fintech_seriesA["Announced Date"])

### 2.1 Collapse to One Round per Startup

For startups with multiple Seed or Series A rounds, retain only the **earliest** round of each type. This ensures one observation per firm in the survival dataset.

In [14]:
# Earliest seed round per startup
seed_first = ( 
    fintech_seed.sort_values("Announced Date")
    .groupby("Organization Name", as_index=False)
    .first()
)

# Earliest Series A per startup 
seriesA_first = (
    fintech_seriesA.sort_values("Announced Date")
    .groupby("Organization Name", as_index=False)
    .first()
)

### 2.2 Merge Seed and Series A — Define Event and Censoring

Left-join seed-funded firms to Series A records. Firms without a Series A match are **right-censored** at the last observed date in the dataset.

In [15]:
# Censor date: last announced date across both datasets
censor_date = max(fintech_seed["Announced Date"].max(), 
                  fintech_seriesA["Announced Date"].max())

In [16]:
# Merge: keep all seed-funded firms, attach Series A if it exists
surv = seed_first.merge(
    seriesA_first[["Organization Name", "Announced Date"]], 
    on= "Organization Name",
    how="left", 
    suffixes=("_seed", "_seriesA")
)
surv.head(3)

,Organization Name,Transaction Name,Transaction Name URL,Organization Location,Organization Name URL,Funding Type,Money Raised,Money Raised Currency,Money Raised (in USD),Announced Date_seed,Equity Only Funding,Organization Revenue Range,Organization Website,Diversity Spotlight,Number of Partner Investors,Number of Investors,Announced Date_seriesA
0,0xCord,Seed Round - 0xCord,https://www.crunchbase.com/funding_round/0xcor...,"New York, New York, United States, North America",https://www.crunchbase.com/organization/0xcord,Seed,25000,USD,25000,2022-04-21,Yes,None,https://0xcord.com,None,NaN,NaN,NaT
1,11th.com,Seed Round - 11th.com,https://www.crunchbase.com/funding_round/11the...,"West Palm Beach, Florida, United States, North...",https://www.crunchbase.com/organization/11thes...,Seed,2000000,USD,2000000,2024-05-29,Yes,$1M to $10M,https://11th.com,None,2.0,5.0,NaT
2,1Token,Seed Round - 1Token,https://www.crunchbase.com/funding_round/1toke...,"New York, New York, United States, North America",https://www.crunchbase.com/organization/1token,Seed,10000000,CNY,1440507,2018-11-19,Yes,None,https://1token.tech/,None,NaN,1.0,2022-06-01


In [17]:
# Rename for clarity
surv = surv.rename(columns={
    "Announced Date_seed": "seed_date", 
    "Announced Date_seriesA": "series_a_date"
})
surv.columns

Index(['Organization Name', 'Transaction Name', 'Transaction Name URL',
       'Organization Location', 'Organization Name URL', 'Funding Type',
       'Money Raised', 'Money Raised Currency', 'Money Raised (in USD)',
       'seed_date', 'Equity Only Funding', 'Organization Revenue Range',
       'Organization Website', 'Diversity Spotlight',
       'Number of Partner Investors', 'Number of Investors', 'series_a_date'],
      dtype='object')

In [18]:
# Event indicator: 1 if Series A exists, 0 otherwise
surv["event_series_a"] = surv["series_a_date"].notna().astype(int)
surv.head()

,Organization Name,Transaction Name,Transaction Name URL,Organization Location,Organization Name URL,Funding Type,Money Raised,Money Raised Currency,Money Raised (in USD),seed_date,Equity Only Funding,Organization Revenue Range,Organization Website,Diversity Spotlight,Number of Partner Investors,Number of Investors,series_a_date,event_series_a
0,0xCord,Seed Round - 0xCord,https://www.crunchbase.com/funding_round/0xcor...,"New York, New York, United States, North America",https://www.crunchbase.com/organization/0xcord,Seed,25000,USD,25000,2022-04-21,Yes,None,https://0xcord.com,None,NaN,NaN,NaT,0
1,11th.com,Seed Round - 11th.com,https://www.crunchbase.com/funding_round/11the...,"West Palm Beach, Florida, United States, North...",https://www.crunchbase.com/organization/11thes...,Seed,2000000,USD,2000000,2024-05-29,Yes,$1M to $10M,https://11th.com,None,2.0,5.0,NaT,0
2,1Token,Seed Round - 1Token,https://www.crunchbase.com/funding_round/1toke...,"New York, New York, United States, North America",https://www.crunchbase.com/organization/1token,Seed,10000000,CNY,1440507,2018-11-19,Yes,None,https://1token.tech/,None,NaN,1.0,2022-06-01,1
3,2EX Technology,Seed Round - 2EX Technology,https://www.crunchbase.com/funding_round/2ex-t...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/2ex-te...,Seed,2000000,USD,2000000,2022-10-01,Yes,None,https://2ex.technology,None,NaN,NaN,NaT,0
4,2nd Wallet,Seed Round - 2nd Wallet,https://www.crunchbase.com/funding_round/2ndwa...,"New York, New York, United States, North America",https://www.crunchbase.com/organization/2ndwallet,Seed,90000,USD,90000,2021-08-01,Yes,None,http://2ndwallet.com,"Women Founded, Women Led",NaN,NaN,NaT,0


In [19]:
# Date we stop following the firm (event or censor)
surv["end_date"] = surv["series_a_date"].fillna(censor_date)

In [20]:
# Time to event / censor in days and months
surv["time_to_series_a_days"] = (surv["end_date"] - surv["seed_date"]).dt.days
surv["time_to_series_a_months"] = surv["time_to_series_a_days"] / 30.44
surv.head()

,Organization Name,Transaction Name,Transaction Name URL,Organization Location,Organization Name URL,Funding Type,Money Raised,Money Raised Currency,Money Raised (in USD),seed_date,...,Organization Revenue Range,Organization Website,Diversity Spotlight,Number of Partner Investors,Number of Investors,series_a_date,event_series_a,end_date,time_to_series_a_days,time_to_series_a_months
0,0xCord,Seed Round - 0xCord,https://www.crunchbase.com/funding_round/0xcor...,"New York, New York, United States, North America",https://www.crunchbase.com/organization/0xcord,Seed,25000,USD,25000,2022-04-21,...,None,https://0xcord.com,None,NaN,NaN,NaT,0,2024-12-17,971,31.898817
1,11th.com,Seed Round - 11th.com,https://www.crunchbase.com/funding_round/11the...,"West Palm Beach, Florida, United States, North...",https://www.crunchbase.com/organization/11thes...,Seed,2000000,USD,2000000,2024-05-29,...,$1M to $10M,https://11th.com,None,2.0,5.0,NaT,0,2024-12-17,202,6.636005
2,1Token,Seed Round - 1Token,https://www.crunchbase.com/funding_round/1toke...,"New York, New York, United States, North America",https://www.crunchbase.com/organization/1token,Seed,10000000,CNY,1440507,2018-11-19,...,None,https://1token.tech/,None,NaN,1.0,2022-06-01,1,2022-06-01,1290,42.378449
3,2EX Technology,Seed Round - 2EX Technology,https://www.crunchbase.com/funding_round/2ex-t...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/2ex-te...,Seed,2000000,USD,2000000,2022-10-01,...,None,https://2ex.technology,None,NaN,NaN,NaT,0,2024-12-17,808,26.544021
4,2nd Wallet,Seed Round - 2nd Wallet,https://www.crunchbase.com/funding_round/2ndwa...,"New York, New York, United States, North America",https://www.crunchbase.com/organization/2ndwallet,Seed,90000,USD,90000,2021-08-01,...,None,http://2ndwallet.com,"Women Founded, Women Led",NaN,NaN,NaT,0,2024-12-17,1234,40.538765


In [21]:
# Sanity check for no negative times
bad = surv[surv["time_to_series_a_days"] < 0]
print(len(bad))
bad.head()

4


,Organization Name,Transaction Name,Transaction Name URL,Organization Location,Organization Name URL,Funding Type,Money Raised,Money Raised Currency,Money Raised (in USD),seed_date,...,Organization Revenue Range,Organization Website,Diversity Spotlight,Number of Partner Investors,Number of Investors,series_a_date,event_series_a,end_date,time_to_series_a_days,time_to_series_a_months
103,Balance,Seed Round - Balance,https://www.crunchbase.com/funding_round/balan...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/balanc...,Seed,2500000,USD,2500000,2024-06-03,...,None,https://www.balancecash.io/blog,None,NaN,NaN,2022-09-08,1,2022-09-08,-634,-20.827858
182,Canopy,Seed Round - Canopy,https://www.crunchbase.com/funding_round/canop...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/canopy...,Seed,75000,USD,75000,2022-06-27,...,None,https://www.heycanopy.com/,None,NaN,1.0,2021-08-09,1,2021-08-09,-322,-10.578187
513,Level,Seed Round - Level,https://www.crunchbase.com/funding_round/level...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/level-...,Seed,2195000,USD,2195000,2021-09-05,...,None,https://www.trylevel.app,Women Founded,NaN,NaN,2021-04-16,1,2021-04-16,-142,-4.664915
974,Waffle Labs,Seed Round - Waffle Labs,https://www.crunchbase.com/funding_round/waffl...,"New York, New York, United States, North America",https://www.crunchbase.com/organization/waffle...,Seed,5004058,USD,5004058,2021-04-05,...,$1M to $10M,https://www.waffleinsurance.com,None,1.0,3.0,2021-03-16,1,2021-03-16,-20,-0.657030


In [22]:
# Remove firms with negative survival times (Series A recorded before Seed)
surv = surv[surv["time_to_series_a_days"] >= 0].copy()

In [23]:
# Confirm dataset dimensions after removing negative-time observations
surv.count()

Organization Name              1030
Transaction Name               1030
Transaction Name URL           1030
Organization Location          1030
Organization Name URL          1030
Funding Type                   1030
Money Raised                   1030
Money Raised Currency          1030
Money Raised (in USD)          1030
seed_date                      1030
Equity Only Funding            1030
Organization Revenue Range      783
Organization Website           1027
Diversity Spotlight             249
Number of Partner Investors     498
Number of Investors             872
series_a_date                   365
event_series_a                 1030
end_date                       1030
time_to_series_a_days          1030
time_to_series_a_months        1030
dtype: int64

---

## Part 3 — Metro Regions, Covariates & Sector Label

### 3.1 Extract City and State from Organization Location

Parse the comma-delimited `Organization Location` field into separate `city` and `state` columns for metro-region mapping.

### 3.1 Extarct city and state from Organziation Location

In [24]:
# Split the coma-delimited "Organization Location" into components
loc_parts = surv["Organization Location"].str.split(",", expand=True)
print(loc_parts.head())

                 0            1               2               3
0         New York     New York   United States   North America
1  West Palm Beach      Florida   United States   North America
2         New York     New York   United States   North America
3    San Francisco   California   United States   North America
4         New York     New York   United States   North America


In [25]:
# Number of location strings
print(len(loc_parts))

1030


In [26]:
# Assign parsed city and state to the survival dataframe
surv["city"] = loc_parts[0].str.strip()
surv["state"] = loc_parts[1].str.strip()
print(surv.head())

  Organization Name             Transaction Name  \
0            0xCord          Seed Round - 0xCord   
1          11th.com        Seed Round - 11th.com   
2            1Token          Seed Round - 1Token   
3    2EX Technology  Seed Round - 2EX Technology   
4        2nd Wallet      Seed Round - 2nd Wallet   

                                Transaction Name URL  \
0  https://www.crunchbase.com/funding_round/0xcor...   
1  https://www.crunchbase.com/funding_round/11the...   
2  https://www.crunchbase.com/funding_round/1toke...   
3  https://www.crunchbase.com/funding_round/2ex-t...   
4  https://www.crunchbase.com/funding_round/2ndwa...   

                               Organization Location  \
0   New York, New York, United States, North America   
1  West Palm Beach, Florida, United States, North...   
2   New York, New York, United States, North America   
3  San Francisco, California, United States, Nort...   
4   New York, New York, United States, North America   

             

### 3.2 Build City-to-Metro Template

Extract the unique (city, state) pairs and export as a template CSV. This template is manually mapped to metro regions (SFBA, GNYA, LA, Miami, Other) and saved as the lookup file loaded below.

In [27]:
# Extract unique (city, state) pairs for metro-region mapping
cities = surv[["city", "state"]].drop_duplicates().reset_index(drop=True)
cities.head()

,city,state
0,New York,New York
1,West Palm Beach,Florida
2,San Francisco,California
3,Fremont,California
4,Palo Alto,California


In [28]:
# Number of unique city-state pairs
len(cities)

88

In [29]:
# Export blank template for manual metro-region assignment
cities.to_csv(r"..\data\cleaned\fintech_loc_map\fintech_city_to_metro_template.csv", index=False)

### 3.3 Merge Metro Region Mapping

Join the manually-curated city-to-metro lookup back to the survival dataset. The merge uses both `city` and `state` to avoid collisions (e.g., Portland, OR vs. Portland, ME). The `validate="m:1"` flag ensures each city maps to exactly one metro region.

In [30]:
# Load the completed city-to-metro mapping (manually curated from the template above)
city_to_metro = pd.read_csv(r"..\data\cleaned\fintech_loc_map\fintech_city_to_metro.csv")

In [31]:
# sanity: no duplicate (city, state) pairs
dupes = city_to_metro[city_to_metro.duplicated(["city", "state"], keep=False)]
print("Duplicate city+state in mapping:\n", dupes)

Duplicate city+state in mapping:
 Empty DataFrame
Columns: [city, state, metro_region]
Index: []


In [32]:
# Left-join metro region mapping onto the survival dataset (many firms → one metro)
surv = surv.merge(
    city_to_metro,
    on=["city", "state"],
    how="left",
    validate="m:1"
)

In [33]:
# Quick duplicate check on firms
firm_counts = surv["Organization Name"].value_counts()
print(firm_counts[firm_counts > 1].count())

0


### 3.4 Rename Crunchbase Variables

Map raw Crunchbase column names to shorter, analysis-friendly names used in the hazard model.

In [34]:
# Rename Crunchbase's dollar-formatted column to a shorter analysis name
surv["seed_amount_usd"] = surv["Money Raised (in USD)"]

In [35]:
# Carry forward investor count fields under shorter names
surv["num_partner_investors"] = surv["Number of Partner Investors"]
surv["num_investors_total"] = surv["Number of Investors"]

### 3.5 Sector Label and Cohort Year

Add the sector identifier (`FinTech`) and extract the seed year for cohort fixed effects in the hazard model.

In [36]:
# Extract cohort year from seed date for fixed effects in the hazard model
surv["seed_year"] = surv["seed_date"].dt.year

# Label all observations in this dataset as FinTech (will be pooled with AI later)
surv["sector"] = "FinTech"  

In [37]:
# Verify all expected columns are present before final selection
surv.columns

Index(['Organization Name', 'Transaction Name', 'Transaction Name URL',
       'Organization Location', 'Organization Name URL', 'Funding Type',
       'Money Raised', 'Money Raised Currency', 'Money Raised (in USD)',
       'seed_date', 'Equity Only Funding', 'Organization Revenue Range',
       'Organization Website', 'Diversity Spotlight',
       'Number of Partner Investors', 'Number of Investors', 'series_a_date',
       'event_series_a', 'end_date', 'time_to_series_a_days',
       'time_to_series_a_months', 'city', 'state', 'metro_region',
       'seed_amount_usd', 'num_partner_investors', 'num_investors_total',
       'seed_year', 'sector'],
      dtype='object')

---

## Part 4 — Final Dataset

Select the analysis-relevant columns, rename `Organization Name` to `firm_id` for consistency across sector datasets, and export the final survival CSV.

In [38]:
simple_surv = surv[[
    "Organization Name",
    "sector",
    "seed_date",
    "series_a_date",
    "end_date",
    "time_to_series_a_months",
    "event_series_a",
    "metro_region",
    "seed_amount_usd",
    "num_partner_investors",
    "num_investors_total",
    "seed_year"
]].copy()

simple_surv = simple_surv.rename(columns={
    "Organization Name" : "firm_id"
})

simple_surv.head()

,firm_id,sector,seed_date,series_a_date,end_date,time_to_series_a_months,event_series_a,metro_region,seed_amount_usd,num_partner_investors,num_investors_total,seed_year
0,0xCord,FinTech,2022-04-21,NaT,2024-12-17,31.898817,0,GNYA,25000,NaN,NaN,2022
1,11th.com,FinTech,2024-05-29,NaT,2024-12-17,6.636005,0,GMA,2000000,2.0,5.0,2024
2,1Token,FinTech,2018-11-19,2022-06-01,2022-06-01,42.378449,1,GNYA,1440507,NaN,1.0,2018
3,2EX Technology,FinTech,2022-10-01,NaT,2024-12-17,26.544021,0,SFBA,2000000,NaN,NaN,2022
4,2nd Wallet,FinTech,2021-08-01,NaT,2024-12-17,40.538765,0,GNYA,90000,NaN,NaN,2021


In [39]:
# Export the final FinTech survival dataset
simple_surv.to_csv(r"..\data\cleaned\cleaned_fintech\fintech_simple_survival.csv", index=False)